# Speech Coach — GPU Backend Server

**Run this notebook on Colab with a T4 GPU runtime.**

### Quick Start:
1. Runtime → Change runtime type → **T4 GPU**
2. Fill in your ngrok token and Claude proxy URL in the cells below
3. **Run All** (Ctrl+F9)
4. Copy the ngrok URL printed at the bottom → paste in webapp Settings

In [ ]:
#@title 1. Install Dependencies
!pip install -q transformers librosa noisereduce
!pip install -q praat-parselmouth
!pip install -q openai-whisper
!pip install -q opencv-python mediapipe ultralytics
!pip install -q spacy textstat sentence-transformers
!pip install -q speechbrain
!pip install -q language_tool_python
!pip install -q fastapi uvicorn python-multipart pyngrok gdown
!python -m spacy download en_core_web_sm -q
print('\u2705 Dependencies installed')

In [ ]:
#@title 2. Clone / Update Repo
import os, sys

REPO_URL = 'https://github.com/anvay-cpu/voice-analysis-pipeline.git'
REPO_DIR = '/content/voice-analysis-pipeline'

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f'\u2705 Repo ready at {REPO_DIR}')

In [ ]:
#@title 3. Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
    print('\u2705 T4 GPU ready')
else:
    print('\u26a0\ufe0f No GPU — go to Runtime -> Change runtime type -> T4 GPU')

In [ ]:
#@title 4. Fix models & download missing files
#@markdown Downloads pose model, disfluency model, and fixes paths.
import os, shutil
os.chdir('/content/voice-analysis-pipeline')

# --- Pose model (MediaPipe) ---
if not os.path.exists('models/pose_landmarker_lite.task'):
    !wget -q -O models/pose_landmarker_lite.task "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task"
    print(f'\u2705 Pose model downloaded: {os.path.getsize("models/pose_landmarker_lite.task")/1e6:.1f}MB')
else:
    print('\u2705 Pose model: exists')

# --- Filler verifier symlink ---
os.makedirs('models/filler', exist_ok=True)
if not os.path.exists('models/filler/best_model.pt'):
    if os.path.exists('models/filler_verifier/best_model.pt'):
        os.symlink('/content/voice-analysis-pipeline/models/filler_verifier/best_model.pt',
                   'models/filler/best_model.pt')
        print('\u2705 Filler verifier: symlinked')
    else:
        print('\u26a0\ufe0f Filler verifier: not found (regex fallback)')
else:
    print('\u2705 Filler verifier: exists')

# --- Disfluency model from shared Google Drive ---
os.makedirs('models/disfluency', exist_ok=True)
dst = 'models/disfluency/best_model.pt'
if os.path.exists(dst) and os.path.getsize(dst) > 100_000_000:
    print(f'\u2705 Disfluency model: {os.path.getsize(dst)/1e6:.1f}MB')
else:
    FILE_ID = '1zuBRwRG_a3kUsTZytLdl80ml29fuI9E9'
    !gdown {FILE_ID} -O {dst} -q
    if os.path.exists(dst) and os.path.getsize(dst) > 100_000_000:
        print(f'\u2705 Disfluency model downloaded: {os.path.getsize(dst)/1e6:.1f}MB')
    else:
        print('\u26a0\ufe0f Disfluency model: download failed (share file as Anyone with link)')

# --- Ensure frames/uploads dirs exist ---
os.makedirs('data/frames', exist_ok=True)
os.makedirs('data/uploads', exist_ok=True)

# --- Model check ---
print('\n--- Model Check ---')
for name, path in {
    'pose_landmarker': 'models/pose_landmarker_lite.task',
    'hand_landmarker': 'models/hand_landmarker.task',
    'face_landmarker': 'models/face_landmarker.task',
    'gesture_transformer': 'models/gesture_transformer/best_model.pt',
    'facial_emotion': 'models/facial_emotion/best_model.pt',
    'posture_mlp': 'models/posture_mlp/best_model.pt',
    'filler_verifier': 'models/filler_verifier/best_model.pt',
    'disfluency': 'models/disfluency/best_model.pt',
    'vocal_emotion': 'models/vocal_emotion/best_model.pt',
}.items():
    if os.path.exists(path):
        s = os.path.getsize(path)
        print(f'  {name}: \u2705 {s/1e6:.1f}MB' if s > 1000 else f'  {name}: \u26a0\ufe0f {s}B')
    else:
        print(f'  {name}: \u274c MISSING')

In [ ]:
#@title 5. Setup ngrok + Claude Proxy
#@markdown **ngrok token** (for Colab server tunnel): get at https://dashboard.ngrok.com/signup
NGROK_AUTH_TOKEN = '3DqOapfoV8a91YHnDDP9SeQhNxu_2F6Cu7RVfHfCNPgWknAgt' #@param {type:"string"}
#@markdown **Claude proxy URL** (from your Mac): run `python scripts/claude_proxy.py --ngrok` locally
CLAUDE_PROXY_URL = 'https://nathaly-coachable-steely.ngrok-free.dev' #@param {type:"string"}

from pyngrok import ngrok, conf
import os, requests

# ngrok auth
if NGROK_AUTH_TOKEN:
    ngrok.kill()  # kill stale tunnels
    conf.get_default().auth_token = NGROK_AUTH_TOKEN
    print('\u2705 ngrok authenticated')
else:
    print('\u26a0\ufe0f No ngrok token')

# Claude proxy
if CLAUDE_PROXY_URL:
    url = CLAUDE_PROXY_URL.strip().rstrip('/')
    os.environ['CLAUDE_PROXY_URL'] = url
    try:
        r = requests.get(f'{url}/health',
                         headers={'ngrok-skip-browser-warning': 'true'}, timeout=5)
        if r.ok:
            print(f'\u2705 Claude proxy connected: {url}')
        else:
            print(f'\u26a0\ufe0f Proxy status {r.status_code} — LLM features will use heuristics')
    except Exception as e:
        print(f'\u26a0\ufe0f Claude proxy unreachable: {e}')
else:
    print('\u2139\ufe0f No Claude proxy — LLM features will use heuristic fallbacks')

In [ ]:
#@title 6. Pre-load Models (warm up GPU)
import os, sys
os.chdir('/content/voice-analysis-pipeline')
if '/content/voice-analysis-pipeline' not in sys.path:
    sys.path.insert(0, '/content/voice-analysis-pipeline')

print('Loading master pipeline...')
from src.master_pipeline import SpeechCoachPipeline
master = SpeechCoachPipeline()

print('Loading voice pipeline...')
try:
    from src.pipeline import VoiceAnalysisPipeline
    master._voice_pipeline = VoiceAnalysisPipeline()
    print('  \u2705 Voice pipeline ready')
except Exception as e:
    print(f'  \u26a0\ufe0f Voice pipeline: {e}')

print('Loading body pipeline...')
try:
    from src.body.pipeline import BodyAnalysisPipeline
    master._body_pipeline = BodyAnalysisPipeline()
    print('  \u2705 Body pipeline ready')
except Exception as e:
    print(f'  \u26a0\ufe0f Body pipeline: {e}')

print('Loading content pipeline...')
try:
    from src.content.pipeline import ContentAnalysisPipeline
    master._content_pipeline = ContentAnalysisPipeline()
    print('  \u2705 Content pipeline ready')
except Exception as e:
    print(f'  \u26a0\ufe0f Content pipeline: {e}')

from src.api_server import set_pipeline
set_pipeline(master)

print('\n\u2705 All models loaded and injected into API server')

In [ ]:
#@title 7. Start GPU Backend Server
#@markdown This cell runs forever — the server stays up as long as Colab is connected.

import threading, asyncio, time, subprocess, os, sys
import uvicorn
from pyngrok import ngrok

os.chdir('/content/voice-analysis-pipeline')
if '/content/voice-analysis-pipeline' not in sys.path:
    sys.path.insert(0, '/content/voice-analysis-pipeline')

from src.api_server import app

# Kill any existing server on port 8000
subprocess.run(['fuser', '-k', '8000/tcp'], capture_output=True)
time.sleep(1)

# Kill stale ngrok tunnels
ngrok.kill()
time.sleep(1)

# Create fresh ngrok tunnel
public_url = ngrok.connect(8000, 'http')
public_url_str = str(public_url).split('"')[1] if '"' in str(public_url) else str(public_url)

print('=' * 60)
print('  SPEECH COACH GPU BACKEND IS LIVE')
print('=' * 60)
print(f'  Public URL: {public_url_str}')
print(f'  Paste in webapp Settings -> COLAB_BACKEND_URL')
print('=' * 60)

def run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='info')
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print('Server stopped')